# 30  Land Registry property, for the dashboard

**What this produces.** Two CSV files that Vishal's dashboard build can read without any
changes to its design, plus a third in the shared signal shape:

```
land_registry_events.csv             one row per property title
land_registry_company_features.csv   one row per company
land_registry_signals.csv            the same events in the eight column signal shape
```

**Numbering.** Viktor now owns 01 to 20 and Vishal owns nb11, so our source notebooks
start at 30. His `20_dashboard_handover.ipynb` landed on 10 August and collided with the first
version of this one.

**What changed from notebook 08.** Two things.

1. Notebook 08 kept only companies that were already in our local spine. That spine is the
   11 June vintage with 869,043 companies, and the dashboard holds 1,493,972. Filtering on
   the old spine threw away most of what the dashboard could use. This notebook emits every
   company number the Land Registry file names, and the dashboard join decides what to keep.
2. It writes files rather than rows in a database, because files are what the dashboard reads.

**Why property is the source to run first.** The Land Registry commercial file carries
`Company Registration No.` in the data itself. No name matching, no postcode confirmation,
no confidence tiers. Every match is exact, which makes this the cleanest source we have and
the only one that needs nothing from the spine rebuild.

**Licence.** Contains HM Land Registry data, Crown copyright and database right. Free to use
under the Land Registry licence, attribution required. That sentence goes in the report.

## 1. Setup

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import io, re, zipfile, getpass

import pandas as pd
import requests

# Colab or local: decide where the working folder is.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/Lloyds")
else:
    WORK_DIR = Path("..").resolve() / "data" / "processed"

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = WORK_DIR / "dashboard_pack"        # everything Vishal gets lands here
OUT_DIR.mkdir(parents=True, exist_ok=True)

# The month the whole dashboard describes. Viktor's handover parquet uses base_month
# 2026-07-01, so we use the same date and our 12 month windows line up with his *_12m
# columns. Vishal's store still says 2026-06-30; that moves when his July crawl lands.
SNAPSHOT_DATE = "2026-07-01"

CCOD_MAX_ROWS = None      # None reads the whole file. Set 200_000 for a quick trial run.

print("work dir :", WORK_DIR)
print("out dir  :", OUT_DIR)
print("snapshot :", SNAPSHOT_DATE)

### The shared helper

`dashboard_export.py` holds the company number cleaner, the date parser and the two file
writers. Every source notebook imports the same copy so the files line up.

**One manual step:** copy `src/dashboard_export.py` from the repo into your Drive folder
(the same folder as `lloyds.duckdb`). The cell below then finds it.

In [ ]:
import sys

CANDIDATES = [WORK_DIR, WORK_DIR / "src", Path("..").resolve() / "src", Path("src").resolve()]
for c in CANDIDATES:
    if (c / "dashboard_export.py").exists():
        sys.path.insert(0, str(c))
        print("helper found in:", c)
        break
else:
    raise FileNotFoundError(
        "dashboard_export.py not found. Copy it from the repo (src/dashboard_export.py) "
        f"into {WORK_DIR} and run this cell again."
    )

from dashboard_export import (clean_company_number, to_iso_date, write_source_pack,
                              EVENT_COLS)
print("helper loaded")

## 2. Getting the file

Land Registry publish **CCOD**, Commercial and Corporate Ownership Data: every freehold and
leasehold title in England and Wales held by a UK company. It is about 3.5 million rows.

Two ways to get it, and the cell below tries both.

**With an API key.** Register free at
<https://use-land-property-data.service.gov.uk/>, accept the licence, and the key appears on
your account page. The cell asks for it, lists the available files, and downloads the newest
FULL one.

**Without a key.** Download the FULL zip from that site by hand, drop it in the working
folder, and press enter when the cell asks for the key. It will find the file.

There is a matching overseas file, OCOD. We are not using it: those owners are not UK
companies, so they have no Companies House number to join on.

In [ ]:
API_BASE = "https://use-land-property-data.service.gov.uk/api/v1"


def find_local_ccod(work_dir):
    """Any CCOD file already sitting in the folder, zip or csv, newest name last."""
    hits = sorted(list(work_dir.glob("CCOD_FULL*.csv")) + list(work_dir.glob("CCOD_FULL*.zip"))
                  + list(work_dir.glob("*ccod*.csv")) + list(work_dir.glob("*ccod*.zip")))
    return hits[-1] if hits else None


def download_ccod(work_dir, key):
    """Ask the API for the newest FULL file and save it. Returns the saved path."""
    head = {"Authorization": key, "Accept": "application/json"}
    r = requests.get(f"{API_BASE}/datasets/ccod", headers=head, timeout=60)
    r.raise_for_status()
    files = r.json().get("result", {}).get("resources", [])
    full = [f for f in files if "FULL" in f.get("file_name", "").upper()]
    if not full:
        raise RuntimeError(f"no FULL file listed. Saw: {[f.get('file_name') for f in files]}")
    name = sorted(f["file_name"] for f in full)[-1]
    print("newest FULL file:", name)

    r2 = requests.get(f"{API_BASE}/datasets/ccod/{name}", headers=head, timeout=60)
    r2.raise_for_status()
    link = r2.json()["result"]["download_url"]

    dest = work_dir / name
    with requests.get(link, stream=True, timeout=600) as resp:
        resp.raise_for_status()
        with open(dest, "wb") as fh:
            for chunk in resp.iter_content(chunk_size=1 << 20):
                fh.write(chunk)
    print(f"saved {dest}  ({dest.stat().st_size / 1e6:.0f} MB)")
    return dest


CCOD_PATH = find_local_ccod(WORK_DIR)
if CCOD_PATH is None:
    key = getpass.getpass("Land Registry API key (blank to skip and use a file you place yourself): ").strip()
    if key:
        CCOD_PATH = download_ccod(WORK_DIR, key)
    else:
        raise FileNotFoundError(
            f"No CCOD file in {WORK_DIR}. Download the FULL zip from "
            "https://use-land-property-data.service.gov.uk/datasets/ccod and put it there."
        )
print("using:", CCOD_PATH)

## 3. Reading it, in chunks

The file is one row per **title**, and a title can have up to four owners, each in its own
set of columns. So we read it, then unfold the four owner slots into one long table of
owner rows.

**Why chunks.** The full file is about 3.5 million titles. Reading it in one go on free
Colab uses several gigabytes and can die part way through with nothing to show. Reading
250,000 rows at a time and keeping only the matched events uses a fraction of that, and it
prints progress so a long run is not a blank screen.

The columns that matter:

| Column | Why |
|---|---|
| `Company Registration No. (1..4)` | the join key, and the reason this source works |
| `Proprietor Name (1..4)` | for the readable line on the screen |
| `Date Proprietor Added` | when the company became the owner, the timeline date |
| `Price Paid` | the value of the event, present on about 4 titles in 10 |
| `Property Address`, `Postcode`, `Tenure` | the detail line, and the postcode for cross checks |

In [ ]:
base_cols = ["Title Number", "Tenure", "Property Address", "Region", "District",
             "County", "Postcode", "Price Paid", "Date Proprietor Added"]
prop_cols = []
for i in (1, 2, 3, 4):
    prop_cols += [f"Proprietor Name ({i})", f"Company Registration No. ({i})",
                  f"Proprietorship Category ({i})"]
wanted = set(base_cols + prop_cols)


def detail_line(r):
    """One readable line for the screen: tenure and address, trimmed."""
    bits = [r.get("Tenure"), r.get("Property Address")]
    txt = ", ".join(str(b).strip() for b in bits
                    if b is not None and str(b).strip() not in ("", "nan"))
    return (txt[:180] or "Property title")


def price(v):
    """Price paid as a number. Blank, nan and text all become None."""
    if v is None or str(v).strip() in ("", "nan"):
        return None
    try:
        return float(re.sub(r"[^0-9.]", "", str(v)) or 0) or None
    except ValueError:
        return None


def unfold(raw):
    """One row per owner. The four owner slots become four stacked frames."""
    frames = []
    for i in (1, 2, 3, 4):
        name_col, num_col = f"Proprietor Name ({i})", f"Company Registration No. ({i})"
        if name_col not in raw.columns:
            continue
        sub = raw[raw[name_col].notna()]
        if sub.empty:
            continue
        out = sub[[c for c in base_cols if c in sub.columns]].copy()
        out["proprietor_name"] = sub[name_col]
        out["reg_no_raw"] = sub[num_col] if num_col in sub.columns else None
        frames.append(out)
    return pd.concat(frames, ignore_index=True) if frames else None

In [ ]:
read_from = CCOD_PATH
if str(CCOD_PATH).lower().endswith(".zip"):
    zf = zipfile.ZipFile(CCOD_PATH)
    inner = [n for n in zf.namelist() if n.lower().endswith(".csv")][0]
    print("reading inside the zip:", inner)
    read_from = zf.open(inner)

CHUNK = 250_000
parts, n_titles, n_owner_rows = [], 0, 0

reader = pd.read_csv(read_from, dtype=str, usecols=lambda c: c.strip() in wanted,
                     nrows=CCOD_MAX_ROWS, na_values=[""], keep_default_na=False,
                     low_memory=False, chunksize=CHUNK)

for k, raw in enumerate(reader, start=1):
    raw.columns = [c.strip() for c in raw.columns]
    n_titles += len(raw)
    owners = unfold(raw)
    if owners is None:
        continue
    n_owner_rows += len(owners)
    owners["CompanyNumber"] = owners["reg_no_raw"].map(clean_company_number)
    m = owners[owners["CompanyNumber"].notna()]
    if m.empty:
        continue
    parts.append(pd.DataFrame({
        "CompanyNumber": m["CompanyNumber"].values,
        "event_date": m["Date Proprietor Added"].map(to_iso_date).values,
        "event_type": "property_title",
        "detail": m.apply(detail_line, axis=1).values,
        "value": m["Price Paid"].map(price).values,
        "url": "",
        "confidence": 1.0,
        "match_method": "company_number",
        "postcode": m["Postcode"].values if "Postcode" in m.columns else None,
        "title_number": m["Title Number"].values if "Title Number" in m.columns else None,
    }))
    print(f"  chunk {k}: {n_titles:,} titles read, "
          f"{sum(len(x) for x in parts):,} matched events so far")

events = pd.concat(parts, ignore_index=True)
print(f"\ntitles read                  : {n_titles:,}")
print(f"owner rows after unfolding   : {n_owner_rows:,}")
print(f"with a usable company number : {len(events):,}  "
      f"({len(events) / max(n_owner_rows, 1):.1%})")
print(f"distinct companies           : {events['CompanyNumber'].nunique():,}")
print(f"with a price paid            : {events['value'].notna().sum():,}  "
      f"({events['value'].notna().mean():.1%})")
events.head(3)

### Titles with no date

`Date Proprietor Added` is blank on a share of older titles. Those rows cannot sit on a
timeline, but the company still owns property and the dashboard should say so. We keep them
in a separate file rather than dropping them, so nothing is lost and the split is visible.

In [ ]:
dated = events[events["event_date"].notna()].copy()
undated = events[events["event_date"].isna()].copy()

print(f"dated   : {len(dated):,} events, {dated['CompanyNumber'].nunique():,} companies")
print(f"undated : {len(undated):,} events, {undated['CompanyNumber'].nunique():,} companies")

only_undated = set(undated["CompanyNumber"]) - set(dated["CompanyNumber"])
print(f"companies we would lose entirely by dropping undated rows: {len(only_undated):,}")

if len(undated):
    undated_path = OUT_DIR / "land_registry_undated_events.csv"
    undated.to_csv(undated_path, index=False)
    print("saved:", undated_path)

## 4. The universe filter

Land Registry covers every UK company that owns property, and most of them are property
companies, housing associations and investment vehicles. On the 200,000 row trial only
**3.8% of the companies found were in our three sectors**, 2,478 of 65,794. Writing the rest
makes a file roughly twenty times bigger than the dashboard can use, and Vishal's build
would discard them on the join anyway.

So we filter on the dashboard universe. It comes from Viktor's handover parquet, which is a
strict superset of Vishal's store: `sector IS NOT NULL` gives the widened universe of
1,505,203 companies at July 2026.

If the parquet is not in the folder yet the cell says so and writes everything, which is
still correct, only larger. Re-run this notebook once the file arrives.

In [ ]:
UNIVERSE = None
bulk_path = WORK_DIR / "dashboard_bulk_2026-07.parquet"

if bulk_path.exists():
    bulk = pd.read_parquet(bulk_path, columns=["CompanyNumber", "sector", "is_active"])
    in_universe = bulk[bulk["sector"].notna()]
    UNIVERSE = set(in_universe["CompanyNumber"].astype(str))
    print(f"universe loaded from {bulk_path.name}")
    print(f"  rows in file       : {len(bulk):,}")
    print(f"  sector IS NOT NULL : {len(UNIVERSE):,}   <- the dashboard universe")
    print(f"  of those, Active   : {int(in_universe['is_active'].sum()):,}")
    hit = set(dated["CompanyNumber"]) & UNIVERSE
    print(f"\n  our property companies inside it: {len(hit):,} "
          f"of {dated['CompanyNumber'].nunique():,}")
else:
    print(f"{bulk_path.name} is not in {WORK_DIR}.")
    print("Ask Viktor for it. Writing every company for now, which is larger but not wrong.")

## 5. Write the pack

In [ ]:
result = write_source_pack(
    events=dated,
    prefix="prop",
    source="land_registry",
    out_dir=OUT_DIR,
    snapshot_date=SNAPSHOT_DATE,
    extra_cols=["postcode", "title_number"],
    universe=UNIVERSE,
)

## 6. What this reached

Two numbers worth writing down for the report: how many companies the source reaches, and
how that compares with the dashboard universe of 1,493,972.

In [ ]:
feats = pd.read_csv(result["features"], dtype={"CompanyNumber": str})
UNIVERSE_N = len(UNIVERSE) if UNIVERSE else 1_505_203   # July widened universe

n = len(feats)
print(f"companies with at least one property title : {n:,}")
print(f"share of the dashboard universe            : {n / UNIVERSE_N:.2%}")
print("the report quotes 5.25% for property, so this is the number to check")
print()
print("titles per company:")
print(feats["prop_count_total"].describe().to_string())
print()
print(f"companies with a title in the last 12 months: {(feats['prop_count_12m'] > 0).sum():,}")
print(f"total price paid recorded                   : "
      f"{feats['prop_value_total'].sum():,.0f} pounds")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))

counts = feats["prop_count_total"].clip(upper=10).value_counts().sort_index()
ax[0].bar(counts.index.astype(str), counts.values, color="#6f52b8")
ax[0].set_title("titles held per company (10+ grouped)")
ax[0].set_xlabel("titles"); ax[0].set_ylabel("companies")

yr = pd.to_datetime(dated["event_date"]).dt.year.value_counts().sort_index()
yr = yr[yr.index >= 2000]
ax[1].plot(yr.index, yr.values, lw=1.6, color="#6f52b8")
ax[1].set_title("titles acquired per year")
ax[1].set_xlabel("year")

plt.tight_layout(); plt.show()

## 7. Done

In `dashboard_pack/`:

- `land_registry_events.csv`
- `land_registry_company_features.csv`
- `land_registry_signals.csv`
- `land_registry_undated_events.csv` if any titles had no date

Next: notebook 31 for trade marks. That one needs a name and postcode index, which now comes
straight from Viktor's `dashboard_bulk_2026-07.parquet` rather than from a spine rebuild.